In [3]:
# Intialization
import os
import sys

os.environ["SPARK_HOME"] = "/home/talentum/spark"
os.environ["PYLIB"] = os.environ["SPARK_HOME"] + "/python/lib"
# In below two lines, use /usr/bin/python2.7 if you want to use Python 2
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.6" 
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"
sys.path.insert(0, os.environ["PYLIB"] +"/py4j-0.10.7-src.zip")
sys.path.insert(0, os.environ["PYLIB"] +"/pyspark.zip")

# NOTE: Whichever package you want mention here.
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0 pyspark-shell' 
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.3 pyspark-shell'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'

In [4]:
#Entrypoint 2.x
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().getOrCreate()

# On yarn:
# spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().master("yarn").getOrCreate()
# specify .master("yarn")

sc = spark.sparkContext

In [4]:
spark.read.parquet(
    "hdfs://localhost:9000/user/talentum/silver_cleaned"
).show(5, False)

+----------------+--------------------+------------------------------------------------------------------------------------------------------------------------------------+------------+--------+-------------------------------+---------------+----------------+---------------------+----------------+-----------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+----------+-------------------------+-------------+----------------+----------------+------------------+-----------+------------------------------------+-------------------------+--------------+-------------+----------------------------------------------------------+----------+-----

In [7]:
bronzeDF.printSchema()

root
 |-- json_data: string (nullable = true)



In [8]:
bronzeDF.show(5, False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [7]:
from pyspark.sql.functions import *

silverDF = spark.read.parquet(
    "hdfs://localhost:9000/user/talentum/silver_cleaned"
)

In [6]:
from pyspark.sql.functions import col, sum, when

silverDF.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in silverDF.columns
]).show(vertical=True, truncate=False)

-RECORD 0---------------------
 Applicants_Count     | 7425  
 Application_Deadline | 26907 
 Benefits             | 6129  
 Company_Name         | 5568  
 Currency             | 2536  
 Education_Level      | 7023  
 Employment_Type      | 59534 
 Experience_Level     | 7170  
 Industry             | 7117  
 Interview_Rounds     | 7345  
 Job_Category         | 14625 
 Job_Description      | 5049  
 Job_ID               | 0     
 Job_Status           | 6163  
 Job_Title            | 5564  
 Location_City        | 7055  
 Location_Country     | 6162  
 Location_State       | 6979  
 Number_of_Openings   | 7291  
 Posted_Date          | 26570 
 Preferred_Skills     | 19468 
 Recruiter_Email      | 9093  
 Recruiter_Name       | 15888 
 Remote_Option        | 7929  
 Required_Skills      | 6502  
 Salary_Max           | 7329  
 Salary_Min           | 7461  
 Selection_Rate       | 22525 
 Source               | 13991 
 Work_Shift           | 6636  



In [8]:
from pyspark.sql.functions import col, sum, when

null_count = silverDF.select([
    sum(when(col(c).isNull(), 1).otherwise(0))
    for c in silverDF.columns
]).collect()[0]

print("Total NULL values:", sum(null_count))

Py4JError: An error occurred while calling z:org.apache.spark.sql.functions.sum. Trace:
py4j.Py4JException: Method sum([class java.util.ArrayList]) does not exist
	at py4j.reflection.ReflectionEngine.getMethod(ReflectionEngine.java:318)
	at py4j.reflection.ReflectionEngine.getMethod(ReflectionEngine.java:339)
	at py4j.Gateway.invoke(Gateway.java:276)
	at py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132)
	at py4j.commands.CallCommand.execute(CallCommand.java:79)
	at py4j.GatewayConnection.run(GatewayConnection.java:238)
	at java.lang.Thread.run(Thread.java:750)



In [9]:
print("Total Rows:", silverDF.count())

Total Rows: 500000


In [10]:
silverDF.printSchema()

root
 |-- Applicants_Count: integer (nullable = true)
 |-- Application_Deadline: date (nullable = true)
 |-- Benefits: string (nullable = true)
 |-- Company_Name: string (nullable = true)
 |-- Currency: string (nullable = true)
 |-- Education_Level: string (nullable = true)
 |-- Employment_Type: string (nullable = true)
 |-- Experience_Level: string (nullable = true)
 |-- Industry: string (nullable = true)
 |-- Interview_Rounds: integer (nullable = true)
 |-- Job_Category: string (nullable = true)
 |-- Job_Description: string (nullable = true)
 |-- Job_ID: string (nullable = true)
 |-- Job_Status: string (nullable = true)
 |-- Job_Title: string (nullable = true)
 |-- Location_City: string (nullable = true)
 |-- Location_Country: string (nullable = true)
 |-- Location_State: string (nullable = true)
 |-- Number_of_Openings: integer (nullable = true)
 |-- Posted_Date: date (nullable = true)
 |-- Preferred_Skills: string (nullable = true)
 |-- Recruiter_Email: string (nullable = true)
 |-

KeyboardInterrupt: 

In [15]:
from pyspark.sql.functions import col

bronzeDF = bronzeDF.filter(
    col("json_data").startswith("{")
)

In [16]:
bronzeDF.show(5, False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [17]:
from pyspark.sql.functions import from_json

sample_json = bronzeDF.select("json_data").first()[0]

print(sample_json)

{"Job_ID": "JOB000001", "Job_Title": "Business Analyst", "Company_Name": "Infosys", "Industry": "Retail", "Job_Category": "Data & Analytics", "Employment_Type": "Part-Time", "Experience_Level": "Executive", "Required_Skills": "Kubernetes|SQL|Go|GraphQL|AWS", "Preferred_Skills": "Flask|Node.js", "Education_Level": "Master's Degree", "Location_City": "Cape Town", "Location_State": "Western Cape", "Location_Country": "South Africa", "Remote_Option": "Yes", "Salary_Min": "415000", "Salary_Max": "762000", "Currency": "ZAR", "Posted_Date": "4/12/2023", "Application_Deadline": "6/25/2023", "Recruiter_Name": "Zara Martin", "Recruiter_Email": "zara.martin@outlook.com", "Job_Description": "We are looking for a talented Business Analyst to join our growing Data & Analytics team at Infosys. You will work on cutting-edge projects, collaborate with cross-functional teams, and drive meaningful impact at scale.", "Benefits": "Home Office Stipend|Commuter Benefits|Dental Insurance", "Work_Shift": "Flex

In [18]:
from pyspark.sql.functions import col

bronzeDF.filter(~col("json_data").startswith("{")).show(20, False)

+---------+
|json_data|
+---------+
+---------+



In [19]:
bronzeDF.count()

1305008

In [20]:
bronzeDF.printSchema()

root
 |-- json_data: string (nullable = true)



In [21]:
bronzeDF.show(10, False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [22]:
bronzeDF.show(10, False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [23]:
bronzeDF.select("json_data").show(10, False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [24]:
from pyspark.sql.functions import regexp_replace

bronzeDF = bronzeDF.withColumn(
    "json_data",
    regexp_replace("json_data", r':\s*NaN', ': null')
)

In [25]:
bronzeDF.show(5, False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [26]:
"Source": null

SyntaxError: illegal target for annotation (<ipython-input-26-fc5982d09a1a>, line 1)

In [27]:
from pyspark.sql.functions import from_json, col

In [28]:
sample = spark.read.json(bronzeDF.rdd.map(lambda x: x.json_data))
schema = sample.schema

In [29]:
silverDF = bronzeDF.withColumn(
    "json",
    from_json(col("json_data"), schema)
).select("json.*")

In [30]:
silverDF.printSchema()

root
 |-- Applicants_Count: string (nullable = true)
 |-- Application_Deadline: string (nullable = true)
 |-- Benefits: string (nullable = true)
 |-- Company_Name: string (nullable = true)
 |-- Currency: string (nullable = true)
 |-- Education_Level: string (nullable = true)
 |-- Employment_Type: string (nullable = true)
 |-- Experience_Level: string (nullable = true)
 |-- Industry: string (nullable = true)
 |-- Interview_Rounds: string (nullable = true)
 |-- Job_Category: string (nullable = true)
 |-- Job_Description: string (nullable = true)
 |-- Job_ID: string (nullable = true)
 |-- Job_Status: string (nullable = true)
 |-- Job_Title: string (nullable = true)
 |-- Location_City: string (nullable = true)
 |-- Location_Country: string (nullable = true)
 |-- Location_State: string (nullable = true)
 |-- Number_of_Openings: string (nullable = true)
 |-- Posted_Date: string (nullable = true)
 |-- Preferred_Skills: string (nullable = true)
 |-- Recruiter_Email: string (nullable = true)
 |

In [31]:
silverDF.show(5, False)

+----------------+--------------------+----------------------------------------------------------------------------------------------------------------------+------------+--------+-----------------+---------------+----------------+------------------+----------------+----------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+----------+-----------------+-------------+----------------+--------------+------------------+-----------+------------------------------+-----------------------+--------------+-------------+-------------------------------------------------------+----------+----------+--------------+----------+----------+
|Applicants_Count|Application_Deadline|Benefits                                                                                                              |Company_Name|Currency|

In [32]:
# Total number of rows
silverDF.count()

1305008

In [33]:
# Total number of columns
len(silverDF.columns)

30

In [34]:
# Display first 20 rows
silverDF.show()

+----------------+--------------------+--------------------+----------------+--------+--------------------+---------------+----------------+--------------------+----------------+----------------+--------------------+---------+----------+--------------------+-------------+----------------+---------------+------------------+-----------+--------------------+--------------------+--------------+-------------+--------------------+----------+----------+--------------+---------------+----------+
|Applicants_Count|Application_Deadline|            Benefits|    Company_Name|Currency|     Education_Level|Employment_Type|Experience_Level|            Industry|Interview_Rounds|    Job_Category|     Job_Description|   Job_ID|Job_Status|           Job_Title|Location_City|Location_Country| Location_State|Number_of_Openings|Posted_Date|    Preferred_Skills|     Recruiter_Email|Recruiter_Name|Remote_Option|     Required_Skills|Salary_Max|Salary_Min|Selection_Rate|         Source|Work_Shift|
+-------------

In [35]:
silverDF.show(20, False)

+----------------+--------------------+----------------------------------------------------------------------------------------------------------------------+----------------+--------+--------------------+---------------+----------------+---------------------+----------------+----------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+----------+------------------------+-------------+----------------+---------------+------------------+-----------+----------------------------------+-------------------------+--------------+-------------+-------------------------------------------------------+----------+----------+--------------+---------------+----------+
|Applicants_Count|Application_Deadline|Benefits                                                                                                        

In [36]:
bronzeDF.count()
silverDF.count()

1305008

In [37]:
silverDF.show(50, False)


+----------------+--------------------+----------------------------------------------------------------------------------------------------------------------+----------------+--------+--------------------------+---------------+----------------+---------------------+----------------+----------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+----------+-------------------------+-------------+----------------+---------------+------------------+-----------+----------------------------------+---------------------------+----------------+-------------+-------------------------------------------------------+----------+----------+--------------+---------------+----------+
|Applicants_Count|Application_Deadline|Benefits                                                                                             

In [38]:
silverDF.show(100, False)

+----------------+--------------------+----------------------------------------------------------------------------------------------------------------------+----------------+--------+--------------------------+---------------+----------------+---------------------+----------------+----------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+----------+-------------------------+-------------+----------------+---------------+------------------+-----------+--------------------------------------+-----------------------------+----------------+-------------+-------------------------------------------------------+----------+----------+--------------+---------------+----------+
|Applicants_Count|Application_Deadline|Benefits                                                                                    

In [39]:
silverDF.printSchema()

root
 |-- Applicants_Count: string (nullable = true)
 |-- Application_Deadline: string (nullable = true)
 |-- Benefits: string (nullable = true)
 |-- Company_Name: string (nullable = true)
 |-- Currency: string (nullable = true)
 |-- Education_Level: string (nullable = true)
 |-- Employment_Type: string (nullable = true)
 |-- Experience_Level: string (nullable = true)
 |-- Industry: string (nullable = true)
 |-- Interview_Rounds: string (nullable = true)
 |-- Job_Category: string (nullable = true)
 |-- Job_Description: string (nullable = true)
 |-- Job_ID: string (nullable = true)
 |-- Job_Status: string (nullable = true)
 |-- Job_Title: string (nullable = true)
 |-- Location_City: string (nullable = true)
 |-- Location_Country: string (nullable = true)
 |-- Location_State: string (nullable = true)
 |-- Number_of_Openings: string (nullable = true)
 |-- Posted_Date: string (nullable = true)
 |-- Preferred_Skills: string (nullable = true)
 |-- Recruiter_Email: string (nullable = true)
 |

In [40]:
from pyspark.sql.functions import count

silverDF.groupBy("Job_ID") \
        .count() \
        .filter("count > 1") \
        .show()

+---------+-----+
|   Job_ID|count|
+---------+-----+
|JOB000154|    4|
|JOB000367|    4|
|JOB000695|    4|
|JOB000889|    4|
|JOB001260|    4|
|JOB001365|    4|
|JOB001414|    4|
|JOB002422|    4|
|JOB002606|    4|
|JOB002640|    4|
|JOB002646|    4|
|JOB002699|    4|
|JOB003425|    4|
|JOB004146|    4|
|JOB004180|    4|
|JOB004310|    4|
|JOB004365|    4|
|JOB004677|    4|
|JOB004716|    4|
|JOB005047|    4|
+---------+-----+
only showing top 20 rows



In [41]:
from pyspark.sql.functions import col, sum, when

silverDF.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in silverDF.columns
]).show()

+----------------+--------------------+--------+------------+--------+---------------+---------------+----------------+--------+----------------+------------+---------------+------+----------+---------+-------------+----------------+--------------+------------------+-----------+----------------+---------------+--------------+-------------+---------------+----------+----------+--------------+------+----------+
|Applicants_Count|Application_Deadline|Benefits|Company_Name|Currency|Education_Level|Employment_Type|Experience_Level|Industry|Interview_Rounds|Job_Category|Job_Description|Job_ID|Job_Status|Job_Title|Location_City|Location_Country|Location_State|Number_of_Openings|Posted_Date|Preferred_Skills|Recruiter_Email|Recruiter_Name|Remote_Option|Required_Skills|Salary_Max|Salary_Min|Selection_Rate|Source|Work_Shift|
+----------------+--------------------+--------+------------+--------+---------------+---------------+----------------+--------+----------------+------------+---------------+

In [42]:
silverDF = bronzeDF.withColumn(
    "json",
    from_json(col("json_data"), schema)
).select("json.*")

In [43]:
silverDF.write \
    .mode("overwrite") \
    .parquet("file:///home/talentum/Downloads/silver")

In [44]:
silverDF.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("file:///home/talentum/Downloads/silver_csv")

In [45]:
silverDF.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("file:///home/talentum/Downloads/silver_csv_single")

----------------------------------------
Exception happened during processing of request from ('127.0.0.1', 40632)
Traceback (most recent call last):
  File "/usr/lib/python3.6/socketserver.py", line 320, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/lib/python3.6/socketserver.py", line 351, in process_request
    self.finish_request(request, client_address)
  File "/usr/lib/python3.6/socketserver.py", line 364, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/lib/python3.6/socketserver.py", line 724, in __init__
    self.handle()
  File "/home/talentum/spark/python/lib/pyspark.zip/pyspark/accumulators.py", line 269, in handle
    poll(accum_updates)
  File "/home/talentum/spark/python/lib/pyspark.zip/pyspark/accumulators.py", line 241, in poll
    if func():
  File "/home/talentum/spark/python/lib/pyspark.zip/pyspark/accumulators.py", line 245, in accum_updates
    num_updates = read_int(self.rfile)
